Ноутбук служит основой для изучения, однако многое нужно будет поискать и почитать самостоятельно. В некоторых заданиях (да и вообще по теме) это дз будет опираться на материал пары.
***

В этом домашнем задании мы (вы) поговорим о, как не странно, **визуализации**. Но говорить о ней надо на каком-то материале. В этот раз немного отдохнем от данных и сами их посоздаем, по моделируем (ну почти).

Я уверена, что вы уже так или иначе владеете кодовыми библиотеками `matplotlib` и `seaborn`, поэтому просто напомню базовый чек-лист из прошлого задания про построение графиков.

- [x] должно быть **понятно**, что изображено на графике;  
- [x] должны присутствовать **названия осей** (за исключением, когда это очевидно);  
- [x] если на графике присутствуют разные данные, то должна быть **легенда**.

В этом дз мы скорее изучим какие-то неочевидные и/или прикольные интересности, чем пройдемся по базовым вещам.  
Для знакомства с базой вот вам [мой любимый простой гайд](https://skillbox.ru/media/code/biblioteka-matplotlib-dlya-postroeniya-grafikov/),
[документация](https://matplotlib.org/3.6.2/users/index.html)
и еще немного материала: [доступен просто так](https://devpractice.ru/matplotlib-lesson-1-quick-start-guide/),
[доступен не просто так](https://pythonworld.ru/novosti-mira-python/scientific-graphics-in-python.html).

# Задание 1: Дискретная непрерывность (2 pt)

Вы знаете, что при $n \rightarrow \infty$ **геометрическое распределение** стремится к **экспоненциальному**
$$\frac{1}{n} Geom\Bigg(\frac{\lambda}{n}\Bigg) \rightarrow Exp(\lambda).$$
Давайте графически это подтвердим!

Для этого нам понадобится:
- [ ] умение [генерировать](####Генерация) случайные величины;
- [ ] умение получать [теоретическую](####Теоретическое-в-коде) плотность;
- [ ] умение [строить](####Построение-графиков) гистограммы;
- [ ] умение [сглаживать](####Сглаживание) данные;
- [ ] умение строить [несколько](####Несколько-графиков) графиков в одной фигуре

*Много теории только тут, дальше столько не будет. Не пропускайте ее, пожалуйста, ну или хотя бы обращайте внимание на выделенное*

#### Генерация

В Python существует **два основных подхода к генерации случайных величин** из различных распределений:  
* `numpy.random`
* `scipy.stats`

В первом случае генерация происходит "на прямую", а во втором как получение выборки конкретной случайной велечины (как метод объекта). Работает `scipy.stats` чуть медленнее, однако это некритично для небольших объемов и точечных вызовов.

Генерация в `numpy.random` происходит через вызов конкретных методов для каждого распределения: `np.random.uniform(a, b, size)`, `np.random.normal(loc, scale, size)`, `np.random.exponential(scale, size)` и так далее.

<div class="alert alert-danger">
<b>ВАЖНО:</b> в экспоненциальном распределении scale = 1/$\lambda$, а не $\lambda$.
</div>

Подробнее как всегда читайте в [документации](https://numpy.org/doc/2.1/reference/random/generator.html), но в этом задании лучше использовать `scipy.stats`.

При этом в обоих случаях не забывайте **фиксировать** `np.random.seed(42)` для **воспроизводимости** результатов!
***

Логика работы со случайными величинами в `scipy.stats` всегда примерно одинаковая:

```python
from scipy import stats

# Создание случайной величины как объекта
xi = stats.norm(loc=0, scale=1)

# Генерация выборки
samples = xi.rvs(size=1000)

pdf_values = xi.pdf(x)      # Плотность вероятности
cdf_values = xi.cdf(x)      # Функция распределения
ppf_values = xi.ppf(q)      # Квантильная функция (обратная CDF)
mean = xi.mean()            # Математическое ожидание
var = xi.var()              # Дисперсия
...
```

При этом в этом модуле реализовано оооооочень много разных распределений!  
| Распределение | Класс | Параметры |
| :------------ | :---- | :-------- |
| **Равномерное** $U(a,b)$ | `stats.uniform` | `loc=a`, `scale=b-a` |
| **Нормальное** $N(\mu,\sigma^2)$ | `stats.norm` | `loc`=$\mu$, `scale`=$\sigma$ |
| **Биномиальное** $Bin(n,p)$ | `stats.binom` | `n`, `p` |
| **Экспоненциальное** $Exp(\lambda)$ | `stats.expon` | `scale`=1/$\lambda$ |
| **Пуассоновское** $Pois(\lambda)$ | `stats.poisson` | `mu`=$\lambda$ |
| **Геометрическое** $Geom(p)$ | `stats.geom` | `p` |
| **Гамма** $Gamma(\alpha,\beta)$ | `stats.gamma` | `a`=$\alpha$, `scale`=1/$\beta$` |
| **Бета** $Beta(\alpha,\beta)$ | `stats.beta` | `a`=$\alpha$, `b`=$\beta$` |
| **Отрицательное биномиальное** $NegBin(r,p)$ | `stats.nbinom` | `n=r`, `p` |
| **Бета-биномиальное** | `stats.betabinom` | `n`, `a`=$\alpha$, `b`=$\beta$` |
| **Стьюдента** $T(d)$ | `stats.t` | `loc=a`, `df=d` |

Как всегда подробнее читайте в [документации](https://docs.scipy.org/doc/scipy/reference/stats.html).

#### Теоретическое в коде


Как было показано ранее из объектов класса распределения можно получать теоретические вещи:
- функцию распределения,
- плотность,
- ожидание,
- дисперсию и так далее
```python
xi = stats.norm(loc=0, scale=1)
cdf_values = xi.cdf(x)
```

Но для дискретных распределений нужно использовать вероятностную массовую функцию (**PMF**), а не плотность (**PDF**). А также их можно получать не от объекта класса, а **через методы сразу**:
```python
binom_pmf = stats.binom.pmf(k, n=100, p=0.05)
```
Для визуализации такого можно использовать `plt.stem()`.

#### Построение графиков

Для построения графиков, очевидно, понадобится `matplotlib.pyplot` или `seaborn`.

Если мы работаем с распределениями, то нам точно понадобятся следующие графики:
| Тип графика | Когда использовать | Пример |
| :---------- | :----------------- | :----- |
| **Гистограмма** (`plt.hist`) | Для отображения эмпирических данных | Приближенная визуализация распределения |
| **Столбчатая диаграмма** (`plt.bar`) | Для дискретных распределений (точное совпадение с целыми значениями) | Визуализация `Pois(λ)` или `Bin(n, p)` без искажения дискретной природы |
| **Линейный график плотности** (`plt.plot`) | Для непрерывных распределений или сглаженных оценок | Наложение теоретической кривой на гистограмму |
| **Точечный график с линиями** (`plt.plot(..., 'o-')`) | Для дискретных распределений при сравнении с теорией | Отображение `pmf` дискретного распределения поверх гистограммы |

**Параметры**

Для лучшей и более понятной визуализации важно уметь работать со следующими параметрами:
| Параметр | За что отвечает | Когда использовать | Пример |
|----------|----------------|-------------------|--------|
| `density=True` | Нормализует гистограмму так, чтобы **площадь = 1** | **Всегда** при сравнении с теоретической плотностью/`pmf` | `plt.hist(data, bins=30, density=True)` |
| `bins` | Количество или границы интервалов гистограммы | Для дискретных распределений: `bins=range(min, max+1)` | `plt.hist(data, bins=range(0, 20))` |
| `alpha` | Прозрачность элемента (0.0–1.0) | При наложении нескольких графиков для видимости перекрытия | `plt.hist(..., alpha=0.6)` |
| `color` / `edgecolor` | Цвет заливки и границ | Для визуального разделения компонентов смеси | `color='skyblue', edgecolor='black'` |
| `label` + `plt.legend()` | Подпись элемента и отображение легенды | При сравнении нескольких распределений на одном графике | `label='Эмпирические данные'` |
| `ax.grid(True, alpha=0.3)` | Добавление сетки | **Всегда** — улучшает читаемость и оценку значений | `ax.grid(True, linestyle='--', alpha=0.3)` |
| `ax.set_xlabel()` / `ax.set_ylabel()` | Подписи осей | **Всегда** — без подписей график непонятен | `ax.set_xlabel('Время между покупками (часы)')` |
| `ax.set_title()` | Заголовок графика | Для краткого описания содержания | `ax.set_title('Сходимость Bin(...) -> Pois(...)')` |
| `ax.annotate()` | Текстовая аннотация со стрелкой | Для выделения ключевых точек (максимумы, пороги) | См. пример ниже |
| `plt.tight_layout()` | Автоматическая настройка отступов | **Перед сохранением** — предотвращает обрезку элементов | `plt.tight_layout()` |
| `dpi` в `savefig()` | Разрешение изображения | Для отчётов: `dpi=150`, для публикаций: `dpi=300` | `plt.savefig('graph.png', dpi=150)` |



Для сравнения данных нам понадобится именно гистограммы (нормированные диаграммы), этого можно добиться при помощи параметра `density=True`: она автоматически нормируется так, чтобы *площадь под диаграммой равнялась 1*.

Подробнее в [документации](https://matplotlib.org/stable/api/pyplot_summary.html).

#### Сглаживание

Когда мы работаем с гистограммой, мы работаем с кусочным объектом, который **зависит от разбиения** (бинов). Поэтому иногда полезно видеть **сглаженную** кривую, которая лучше показывает форму распределения.

Для этой цели можно использовать **ядерное сглаживание** `gaussian_kd`:
* Ядро сглаживания строит гладкую кривую через данные
* Параметр `bw_method` регулирует "гладкость" (по умолчанию — оптимальный)
* Для малых выборок используйте `kde = gaussian_kde(data, bw_method=0.5)`
```python
kde = gaussian_kde(data)
x = np.linspace(min(data), max(data), 1000)
plt.plot(x, kde(x))
```

**Как выбрать** параметр `bw_method`?

```python
# Слишком гладко (теряем детали)
kde1 = gaussian_kde(samples, bw_method=2.0)

# Оптимально (по умолчанию)
kde2 = gaussian_kde(samples, bw_method='scott')  # или 'silverman'

# Слишком "рвано" (шум вместо сигнала)
kde3 = gaussian_kde(samples, bw_method=0.1)
```

<div class="alert alert-danger">
<b>ВАЖНО:</b> Сглаживание может ввести в заблуждение — создать иллюзию непрерывности.
</div>

Подробнее угадайте где? В [документации](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.gaussian_kde.html)!

#### Несколько графиков

Когда нужно сравнить распределения при разных параметрах или показать эволюцию процесса, **одного графика недостаточно**. В этом случае используем **сетку графиков (subplots)**.

**Базовый синтаксис**: `plt.subplots()`

```python
import matplotlib.pyplot as plt

# Создание сетки 2x3 (2 строки, 3 столбца)
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(15, 8))
# axes — это двумерный массив объектов осей

# Доступ к отдельным графикам
ax_top_left = axes[0, 0]    # верхний левый
ax_bottom_right = axes[1, 2]  # нижний правый

# Если нужен одномерный массив (удобнее для циклов)
axes_flat = axes.flatten()  # превращает (2,3) → (6,)
```

**Ключевые параметры**:

| Параметр | За что отвечает | Пример |
| :------- | :-------------- | :----- |
| `nrows`, `ncols` | Количество строк и столбцов |  `plt.subplots(2, 3)` -> сетка 2x3 |
| `figsize=(w, h)` | Размер всей фигуры в дюймах | `figsize=(14, 6)` — широкая фигура для горизонтального сравнения |
| `sharex=True/False` | Общая ось X для всех графиков | При сравнении распределений с одинаковым диапазоном значений |
| `sharey=True/False` | Общая ось Y для всех графиков | При сравнении плотностей или диаграмм с одинаковым масштабом |
| `squeeze=True` *(по умолчанию)* | Упрощает форму массива `axes` | При `nrows=1`, `ncols=1` возвращает один объект `ax`, а не массив 1x1 |
| `gridspec_kw` | Тонкая настройка размеров ячеек | `gridspec_kw={'width_ratios': [2, 1]}` — первый график в 2 раза шире второго |

Ну и подробнее в любимой [документации](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html).

***
## Задание (2 pt)

Изучите приближение геометрического распределения к экспоненциальному через визуализацию.

**Постройте сетку 2×2 графиков для разных значений $n$ (20, 50, 100, 1000) при фиксированном значении $\lambda=2$**  
**На каждом графике** отобразите:
- гистограмму выборки из $\frac{1}{n} Geom\big(\frac{\lambda}{n}\big)$,
- теоретическую плотность соответствующего экспоненциального,
- сглаженную плотность из данных,
- легенду с нужными значениями.

Для этого используйте нужные виды графиков, цветовую дифференциацию, правильные подписи и вообще _**всю красоту**_.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import gaussian_kde

np.random.seed(42)
lambda_param = 2.0
n_values = [10, 50, 100, 500]
n_samples = ...  # размер выборки для каждого n

# Создайте сетку 2x2
fig, axes = plt.subplots(...)
axes = axes.flatten()

for idx, n in enumerate(n_values):
    ax = axes[idx]
    ...
   
    ax.hist(...,)
    
    x_vals = np.linspace(0, 5, 200)
    exp_pdf = ... 
    ax.plot(...)
    
    kde = ...
    ax.plot(...)
    
    ax.set_xlabel(...)
    ax.set_ylabel(...)
    ax.set_title(...)
    ax.legend()
    ax.grid(...)

fig.suptitle(...)
plt.tight_layout()
# plt.savefig('task1_geometric_exponential.png', dpi=150, bbox_inches='tight') МОЖНО СОХРАНИТЬ, А НЕ ПРОСТО ВЫВЕСТИ
plt.show()

# Задание 2: Не детские смеси (3 pt)

В этом задании мы повизуализируем смеси. Все нужые **ссылки и прочее уже были даны ранее**, добавлю только несколько фич:

| Фича | Метод | Зачем использовать |
| :--- | :---- | :----------------- |
| **Аннотации со стрелками** | `ax.annotate()` | Выделение ключевых точек графика с визуальной связью между текстом и объектом |
| **Текстовые блоки с фоном** | `ax.text()` + `bbox` | Добавление поясняющего текста с выделением фона для читаемости |
| **Вертикальные/горизонтальные линии** | `ax.axvline()`, `ax.axhline()` | Выделение порогов, средних значений, квантилей |
| **Заливка областей** | `ax.fill_between()`, `axvspan()` | Визуализация доверительных интервалов, выделение зон интереса |
| **Настройка шрифтов** | `fontdict`, `fontsize` | Единый стиль оформления текстовых элементов |
| **Стили линий и маркеров** | `linestyle`, `marker` | Различение нескольких кривых без цвета |

У всего этого куча интересных параметров, за ними идите в [ДО](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.annotate.html)
[КУ](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.text.html)
[МЕН](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.axvline.html)
[ТА](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.fill_between.html)
[ЦИ](https://matplotlib.org/stable/gallery/text_labels_and_annotations/accented_text.html)
[Ю](https://matplotlib.org/stable/gallery/text_labels_and_annotations/annotation_basic.html)!

## «Теоретическое» задание (1,5 pt)

Смоделируйте **загрязнённое нормальное распределение**, которое можно встретить, например, в ситуации, когда есть основная группа пользователей и малая часть "активных"

$$(1 - \varepsilon) \cdot N(\mu, \sigma^2) + \varepsilon \cdot N(\mu, \alpha\sigma^2).$$

Ваша **необходимо**:
1. Сгенерировать **выборку** из загрязненного нормального распределения с параметрами:
   - $\mu = 0$, $\sigma =  1$
   - $\varepsilon = 0.15$ *(вот тут появляется Бернулли)*
   - $\alpha = 4$
2. Построить **один график** со следующими элементами:
   - Гистограмма эмпирических данных (серый цвет, `alpha=0.6`)
   - Теоретическая плотность основного компонента (синяя линия, `alpha=0.7`)
   - Теоретическая плотность загрязняющего компонента (красная линия, `alpha=0.7`)
   - Полная теоретическая плотность смеси (чёрная пунктирная линия, толщина 2.5)
   - Сравнение с "чистым" нормальным распределением (фиолетовая штрихпунктирная линия) — *чтобы показать, как загрязнение искажает форму*
   - Аннотация со стрелкой, указывающая на "тяжёлые хвосты" от загрязняющего компонента
   - Текстовый блок с параметрами смеси и интерпретацией (используйте `bbox`)

В этом задании для генерации используем `numpy.random`.

In [ ]:
np.random.seed(42)

mu = ...
sigma = ...
epsilon = ...
alpha = ...
n_samples = ...

# Генерация смеси
is_contaminated = np.random.binomial(1, epsilon, n_samples)
main_component = np.random.normal(...)
contaminated_component = np.random.normal(...)
mixture = np.where(is_contaminated == 1, contaminated_component, main_component)

# Создание графика
fig, ax = ...

# Гистограмма
ax.hist(...)

# Теоретические компоненты
x_vals = np.linspace(-8, 8, 500)
main_pdf = ...
ax.plot(...)

contaminated_pdf = ...
ax.plot(...)

mixture_pdf = ...
ax.plot(...)

# Чистое нормальное для сравнения
pure_normal_pdf = ...
ax.plot(...)

# Аннотация на тяжёлые хвосты
ax.annotate(...)

# Текстовый блок с интерпретацией
interpretation_text = (...)
ax.text(...)

# Оформление
ax.set_xlabel(...)
ax.set_ylabel(...)
ax.set_title(...)
ax.legend(...)
ax.grid(...)

plt.tight_layout()
# plt.savefig('task2_contaminated_normal.png', dpi=150, bbox_inches='tight')
plt.show()

## Практическое задание (1,5 pt)

В этом задании мы все же поработаем с данными, но с теми, с которыми вы уже знакомы! Будем снова говорить про пингвинчиков)

Вам **необходимо**:
1. Загрузить датасет и удалить строки с пропущенными значениями в столбце `bill_length_mm`
2. Построить **два графика** на **одной фигуре** (сетка 1×2):
   - Левый график: **Общая гистограмма** длины клюва всех пингвинов + наложенные плотности для каждого вида (разные цвета, `alpha=0.6`)
   - Правый график: **Три отдельные гистограммы** по видам (на одном графике для сравнения) с **теоретическими нормальными кривыми**
3. На левом графике добавьте:
   - **Аннотацию со стрелкой на бимодальность** (место между пиками)
   - **Текстовый блок с интерпретацией**: почему общее распределение выглядит как смесь?
   - **Вертикальные линии**, показывающие средние значения для каждого вида
4. На правом графике добавьте:
   - **Легенду** с указанием вида и количества особей
   - **Подпись с выводом**: *какой вид имеет наибольшую вариабельность длины клюва?*

In [ ]:
penguins = sns.load_dataset('penguins')
penguins_clean = penguins.dropna(subset=['bill_length_mm'])

adelie = ...
chinstrap = ...
gentoo = ...
all_penguins = ...

# Вычисление параметров
mu_adelie, std_adelie = ...
mu_chinstrap, std_chinstrap = ...
mu_gentoo, std_gentoo = ...

# Создание графиков
fig, axes = ...

# Левый график: общая смесь
ax1 = axes[0]
...

# Правый график: сравнение видов
ax2 = axes[1]
...

plt.tight_layout()
# plt.savefig('task2_penguins_mixture.png', dpi=150, bbox_inches='tight')
plt.show()

# Задание 3: Buy Till You Die (5 pt)

В этом задании мы поработаем с моделью, которая может пригодиться при аналитике реальных процессов. Напоминаю:

> Предположим, что после первой покупки, пользователи ведут себя так:  
> С вероятностью `𝑝` он перестает пользоваться сервисом, а с вероятностью `1 − 𝑝` не перестает.
> Тем самым количество покупок имеет распределение `𝐺𝑒𝑜𝑚(𝑝)`. Такой способ моделировать «смерть» пользователя называется **Buy Till You Die (BTYD)**  
> Если покупатель не перестал пользоваться сервисом, то следующая покупка происходит через время `𝐸𝑥𝑝(𝜆)`  
> У каждого пользователя параметры `𝑝` и `𝜆` свои, и **вариабельность в их значениях** моделируются с помощью `𝐵𝑒𝑡𝑎(𝛼𝑝,𝛽𝑝)` для `𝑝` и `𝐺𝑎𝑚𝑚𝑎(𝛼𝜆, 𝛽𝜆)` для `𝜆`.  
> Получившуюся модель называют **Beta-Geometric/Negative Binomial Distribution (BG/NBD)**.


### Теоретическое задание (1,5 pt)

Пусть в момент последней активности пользователь с вероятностью $p$ ушел, и, стало быть, с вероятностью $1−p$ остался. Если пользователь остался, то его следующая активность случится через время $T$, которое имеет распределение $Exp(q)$.

Выразите через $p$, $q$ и $t$ вероятность того, что **пользователь остался при условии, что он не проявлял активность в течение времени** $t$.

В КАЧЕСТВЕ РЕШЕНИЯ ОЖИДАЕТСЯ ФОРМУЛА И ТО, КАК ВЫ ЕЕ ПОЛУЧИЛИ (С ПОЯСНЕНИЕМ РАССУЖДЕНИЙ)

### Практическое задание (3,5 pt)

Будем использовать публичный датасет `CDNOW` — классический датасет для анализа поведения покупателей (покупки музыки онлайн в 1997-98 годах). Датасет содержит:
- `customer_id` — идентификатор покупателя
- `date` — дата покупки
- `quantity` — количество купленных альбомов
- `value` — сумма покупки в долларах

Ваша задача: **построить график** 2x2, отвечающий на **4 бизнес-вопроса**:

| Подграфик | Вопрос | Требования |
| :-------- | :----- | :--------- |
| **Верхний левый** | Как распределено количество покупок на пользователя? | Гистограмма с `bins=range(1, max_quantity+1)` <br> Теоретическая кривая геометрического распределения <br> Аннотация с указанием доли «одноразовых» покупателей |
| **Верхний правый** | Как распределено время между покупками? | Гистограмма с `density=True` <br> Теоретическая кривая экспоненциального распределения <br> Вертикальная линия на медиане времени между покупками |
| **Нижний левый** | Какова вероятность «смерти» пользователя в зависимости от времени с последней покупки? | Кривая функции выживания `S(t)=P(T>t)` <br> Кривая «риска смерти» `h(t)=1−S(t)` <br> Заливка области под кривой риска (`fill_between`) |
| **Нижний правый** | Сколько покупок совершит пользователь за следующие 30 дней? | Гистограмма прогноза (сгенерированная через симуляцию) <br> Вертикальная линия на математическом ожидании прогноза <br> Доверительный интервал 95% через заливку (`axvspan`) |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

df = pd.read_csv('cdnow.csv', index_col=0)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['customer_id', 'date']).reset_index(drop=True)

...

**НАПИШИТЕ ВЫВОДЫ ПО ГРАФИКАМ, КОТОРЫЕ БЫ ВЫ СДЕЛАЛИ, КАК АНАЛИТИК (ОТВЕТЫ НА ВОПРОСЫ + РАССУЖДЕНИЯ)**

# Бонус с собеса (2+ pt)

Мне понравилась задачка с собеседования, поэтому показываю ее вам:

> На маркетплейсе есть $n$ категорий товаров, в каждой из которых есть $n_i$ различных позиций товаров. Пользователь собирает корзину, делая покупки из любых категорий (может взять из категории, может не взять, может взять из нескольких). Известно, что **средний чек по каждой из категорий** вырос за последний месяц. Что могло произойти с **общим средним чеком**?

Сначала поймите, как считается общий чек. В рассуждениях вы свободны, главное оформите так, чтобы я могла _**понять, где ваш ответ и выводы, на чем они основаны**_. Вы можете делать теоретические выкладки, симуляции и визуализацию. Ничего из этого не запрещено.